In [1]:
import pandas as pd
import os


def fpkm(data):
    """
    对txt进行操作,将count值转为RPKM值
    """
    columns = data.columns # 存表的列名
    table = data.iloc[:, :6] # 读取前6列

    for i in range(data.shape[1]-6): 
        reads = data.iloc[:, i+6] 
        total_counts = reads.sum()/1000000
        RPM = reads/total_counts
        fpkm_result = RPM*1000/(data['Length']) # Length是表中的某一列，是基因的长度
        table = pd.concat([table, fpkm_result], axis=1)

    table.columns = columns
    return table


def txt_to_csv(txt_read_path, csv_output_path):
    """
    Description: 批量将txt文件转为csv文件
    """
    try:
        for file_name in os.listdir(txt_read_path):

            data = pd.read_table(txt_read_path + "/" + file_name, sep='\t', header=1)
            file_name = file_name.split('.')[0] + '.csv'
            data.to_csv(csv_output_path + "/" + file_name, sep=',' ,index=False)

    except Exception as e:
        print(e)
        return False


def txt_to_csv_fpkm(txt_read_path, csv_output_path):
    """
    Description: 批量将txt文件转为csv文件，并处理为fpkm
    """
    try:
        for file_name in os.listdir(txt_read_path):

            data = pd.read_table(txt_read_path + "/" + file_name, sep='\t', header=1)
            RPKM_data = fpkm(data)
            file_name = file_name.split('.')[0] + '.csv'
            RPKM_data.to_csv(csv_output_path + "/" + file_name, sep=',' ,index=False)

    except Exception as e:
        print(e)
        return False


def merge_csv_col(first_csv_path, csv_output_path):
    """
    将输出目录中所有的csv文件的第7列数据进行合并，并返回合并后的表格数据
    """
    first = pd.read_csv(first_csv_path, sep=',')
    New_table = first.iloc[:,0:6] # 取出第一个CSV文件前六列，用于创建新表

    for file_name in os.listdir(csv_output_path):
        if file_name.endswith('.csv'):
            data = pd.read_csv(csv_output_path + file_name, sep=',')
            New_table = pd.concat([New_table, data.iloc[:,6]], axis=1) # 将所有csv的第7列拼接到一个表的基因号后面

    return New_table

In [2]:
txt_to_csv_fpkm('./Fungi/Mt/Counts/', './Fungi/Mt/FPKM/')

In [3]:
df_merge = merge_csv_col('./Fungi/Mt/FPKM/SRR10099850.csv','./Fungi/Mt/FPKM/')
df = df_merge.rename(columns=lambda x: x.replace(".bam","").replace('"',''))
df

,Geneid,Chr,Start,End,Strand,Length,SRR21608676,SRR21608668,SRR101401,SRR21702079,...,SRR22838477,SRR6854517,SRR15712860,SRR21608675,SRR7121126,SRR15712841,SRR1267421,SRR22838486,SRR21608663,SRR15712820
0,MYCTH_2114025,NC_016472.1,1381,2142,+,762,2.133204,3.027872,1.786545,3.204380,...,4.765069,0.824609,1.953426,2.409430,1.484038,14.222427,1.540085,2.834953,4.925344,5.323856
1,MYCTH_2293935,NC_016472.1,2744,3343,-,600,0.550300,0.081817,0.523595,1.102173,...,4.606470,0.209451,0.000000,0.754515,0.819447,2.690157,1.086616,0.439072,1.787196,0.689928
2,MYCTH_2293936,NC_016472.1,3344,3817,-,474,0.482249,0.051783,1.104631,0.107320,...,8.574964,0.318153,0.000000,0.795902,0.414910,0.000000,0.550185,1.000417,1.084006,1.047992
3,MYCTH_2051335,NC_016472.1,5330,6385,-,1056,7.071164,3.579493,10.809065,13.993937,...,22.974190,5.355274,35.130933,8.264413,6.564888,57.264096,12.224426,16.639831,13.095105,44.610136
4,MYCTH_2121898,NC_016472.1,15458,25533,+,10076,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.002494,0.000000,0.000000,0.000000,0.000000,0.012941,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9287,MYCTH_103797,NC_016478.1,4089438,4091337,+,1900,0.053470,0.103348,0.716499,0.160641,...,5.238731,1.388989,1.868171,0.105897,0.284650,57.039418,3.019648,0.471425,8.124688,10.370712
9288,MYCTH_2071098,NC_016478.1,4095568,4097284,+,1717,6.360711,2.730409,4.513225,12.058181,...,9.721422,5.152707,17.138445,6.342553,8.905580,24.676746,10.480113,9.911724,11.462725,8.631135
9289,MYCTH_2114022,NC_016478.1,4098122,4098907,-,786,0.000000,0.000000,0.532921,0.000000,...,0.000000,0.000000,0.291351,0.000000,0.000000,0.000000,0.497687,0.000000,0.113689,0.210665
9290,MYCTH_2114023,NC_016478.1,4100606,4101293,-,688,1.070575,0.285408,7.001562,0.295753,...,1.312834,0.255725,2.995661,0.475227,0.357317,0.000000,0.947630,1.148735,0.876713,0.240673


In [4]:
# 保存
df.to_csv('./Fungi/Mt/Mt_FPKM_Run221.csv', sep=',', index=False)

# 对count进行合并

In [23]:
txt_to_csv('./Nc/Counts/', './Nc/Counts_csv/')
df_merge_counts = merge_csv_col('./Nc/Counts_csv/SRR10078920.csv','./Nc/Counts_csv/')

In [26]:
df_counts = df_merge_counts.rename(columns=lambda x: x.replace(".bam","").replace('"',''))
df_counts.to_csv('./Nc/Nc_Counts_1132Run.csv', sep=',', index=False)